# Do bronze pra Silver

In [1]:
import os
import re
import unidecode
import glob
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, StringType, FloatType, TimestampType


# --- Função de Processamento para cada Arquivo ---
# Esta função será distribuída e executada em paralelo pelo Spark para cada arquivo.
def processar_conteudo_arquivo(file_info):
    """
    Processa o conteúdo de um único arquivo CSV, extraindo metadados do cabeçalho
    e os dados principais.
    
    Args:
        file_info (tuple): Uma tupla contendo (caminho_do_arquivo, conteudo_completo_do_arquivo).

    Returns:
        list: Uma lista de dicionários, onde cada dicionário representa uma linha de dados
              enriquecida com os metadados do arquivo.
    """
    caminho_arquivo, conteudo_completo = file_info
    linhas = conteudo_completo.split('\n')
    
    # 1. Extração de Metadados do Cabeçalho (primeiras 8 linhas)
    metadados = {}
    cabecalho_linhas = linhas[:8]
    for linha in cabecalho_linhas:
        if ';' in linha:
            partes = linha.split(';', 1)
            chave = partes[0].strip()
            valor = partes[1].strip() if len(partes) > 1 else None
            
            # Limpeza e normalização da chave
            chave_limpa = unidecode.unidecode(chave).upper()
            chave_limpa = re.sub(r'[^A-Z0-9 ]', '', chave_limpa)
            if 'REGIO' in chave_limpa: chave_limpa = 'REGIAO'
            if 'ESTACO' in chave_limpa: chave_limpa = 'ESTACAO'
            
            metadados[chave_limpa] = valor

    # 2. Extração de Dados Tabulares (a partir da linha 9)
    registros = []
    dados_linhas = linhas[9:] # A linha 8 é o header dos dados
    
    for linha_dado in dados_linhas:
        valores = linha_dado.strip().split(';')
        # Garante que a linha tem o número mínimo de colunas para evitar erros
        if len(valores) >= 11 and valores[0]: # Aumentado para 11 para garantir a coluna de temperatura
            try:
                # O nome da coluna de data pode variar
                data = valores[0]
                hora = valores[1]
                temperatura = valores[8] 
                precipitacao = valores[2]
                umidade = valores[15]
                vento = valores[18]
                # Monta um dicionário para a linha de dados
                registro = {
                    'Regiao': metadados.get('REGIAO'),
                    'UF': metadados.get('UF'),
                    'Estacao': metadados.get('ESTACAO'),
                    'Latitude': float(str(metadados.get('LATITUDE', '0')).replace(',', '.')),
                    'Longitude': float(str(metadados.get('LONGITUDE', '0')).replace(',', '.')),
                    'Altitude': float(str(metadados.get('ALTITUDE', '0')).replace(',', '.')),
                    'DataOriginal': data,
                    'HoraOriginal': hora,
                    'TemperaturaOriginal': temperatura,
                    'PrecipitacaoOriginal': precipitacao,
                    'UmidadeOriginal': umidade,
                    'VentoOriginal': vento
                }
                registros.append(registro)
            except (IndexError, ValueError) as e:
                # Ignora linhas malformadas, opcionalmente pode-se logar o erro
                print(f"Linha ignorada por erro: {e} -> {linha_dado}")
                pass
                
    return registros

# --- Script Principal PySpark ---
def main():
    spark = SparkSession.builder \
        .appName("Processamento INMET com PySpark") \
        .master("spark://spark-master:7077") \
        .getOrCreate()

    try:
        # Ajuste o caminho para ser mais flexível, se necessário
        caminho_base = "../data/bronze/INMET/*"
        pasta_destino = "../data/silver/INMET_PARQUET"
        caminho_leitura = f"{caminho_base}" # Lê de todos os anos

        # Usamos sorted para garantir uma ordem de processamento consistente
        lista_de_pastas = sorted(glob.glob(caminho_base))
        
        if not lista_de_pastas:
            print(f"Nenhuma pasta encontrada no caminho: {caminho_base}")
            return

        # 2. Definir o tamanho do lote e iterar sobre as pastas
        tamanho_lote = 3
        modo_salvar = "overwrite" # O primeiro lote sempre sobrescreve

        for i in range(0, len(lista_de_pastas), tamanho_lote):
            lote_pastas = lista_de_pastas[i:i + tamanho_lote]
            
            num_lote_atual = (i // tamanho_lote) + 1
            num_total_lotes = (len(lista_de_pastas) + tamanho_lote - 1) // tamanho_lote
            
            print(f"\n--- Processando Lote {num_lote_atual}/{num_total_lotes} ---")
            print(f"Pastas neste lote: {lote_pastas}")

            # 3. Construir o caminho de leitura para o lote atual
            # Spark pode ler de múltiplos caminhos separados por vírgula
            caminhos_lote = [f"{pasta}/*.[cC][sS][vV]" for pasta in lote_pastas]
            caminho_leitura = ",".join(caminhos_lote)

            rdd_arquivos = spark.sparkContext.wholeTextFiles(caminho_leitura)
            
            # Se o lote não contiver arquivos CSV, pule para o próximo
            if rdd_arquivos.isEmpty():
                print("Nenhum arquivo .CSV encontrado neste lote. Pulando para o próximo.")
                continue

            rdd_processado = rdd_arquivos.flatMap(processar_conteudo_arquivo)
            rdd_processado.persist()

            if rdd_processado.isEmpty():
                print("Nenhum dado foi processado neste lote. Verifique o conteúdo dos arquivos.")
                continue

            # --- SCHEMA EXPLÍCITO ---
            schema_definido = StructType([
                StructField("Regiao", StringType(), True),
                StructField("UF", StringType(), True),
                StructField("Estacao", StringType(), True),
                StructField("Latitude", FloatType(), True),
                StructField("Longitude", FloatType(), True),
                StructField("Altitude", FloatType(), True),
                StructField("DataOriginal", StringType(), True),
                StructField("HoraOriginal", StringType(), True),
                StructField("TemperaturaOriginal", StringType(), True),
                StructField("PrecipitacaoOriginal", StringType(), True),
                StructField("UmidadeOriginal", StringType(), True),
                StructField("VentoOriginal", StringType(), True)
            ])
    
            # Cria o DataFrame fornecendo o schema para evitar inferência.
            df = spark.createDataFrame(rdd_processado, schema=schema_definido)
    
            #df.show(10,truncate=False)
    
            df_transformado = (
                df
                .withColumn('Temperatura', sf.regexp_replace(sf.col('TemperaturaOriginal'), ',', '.').try_cast(FloatType()))
                .withColumn('Precipitacao', sf.regexp_replace(sf.col('PrecipitacaoOriginal'), ',', '.').try_cast(FloatType()))
                .withColumn('Umidade', sf.regexp_replace(sf.col('UmidadeOriginal'), ',', '.').try_cast(FloatType()))
                .withColumn('Vento', sf.regexp_replace(sf.col('VentoOriginal'), ',', '.').try_cast(FloatType()))
                .withColumn("DataHoraOriginal", 
                        sf.regexp_replace(
                            sf.concat_ws(" ", sf.col("DataOriginal"), sf.col("HoraOriginal")),
                            " UTC",  # O texto a ser encontrado (note o espaço antes de UTC)
                            ""       # O texto pelo qual substituir (uma string vazia para remover)
                        )
                    )
                .withColumn('_data_timestamp',sf.coalesce(
                    sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy-MM-dd HHmm')),
                    sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy-MM-dd HH:mm')),
                    sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy/MM/dd HHmm')),
                    sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy/MM/dd HH:mm'))
                            )
                           )
                .filter(sf.col('_data_timestamp').isNotNull())
                .filter(sf.col('Temperatura') != -9999.0)
                .withColumn('Data', sf.date_format(sf.col('_data_timestamp'), 'yyyy-MM-dd'))
                .withColumn('Tempo', sf.date_format(sf.col('_data_timestamp'), 'HH:mm'))
                .withColumn('Ano', sf.year(sf.col('_data_timestamp')))
                .select('Data', 'Tempo', 'Temperatura', 'Precipitacao', 'Umidade', 'Vento','Regiao', 'UF', 'Estacao', 'Latitude', 'Longitude', 'Altitude', 'Ano')
            )
    
            #df_transformado.show(10,truncate=False)
            print(f"Salvando os dados transformados em {pasta_destino}...")
            df_transformado.write \
                .partitionBy("Ano") \
                .mode(modo_salvar) \
                .parquet(pasta_destino)
    
            
            modo_salvar = "append"
                
                # Libera o RDD da memória para o próximo lote
            rdd_processado.unpersist()
    
        print("Processamento concluído com sucesso!")

    finally:
        print("Encerrando a sessão Spark.")
        spark.stop()

if __name__ == '__main__':
    main()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/02 23:58:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Enriquecimento dos dados transformando Estações em cidades.

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Leitura INMET com PySpark") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

caminho_clima = "../data/silver/INMET_PARQUET"

df_clima = spark.read.parquet(caminho_clima)


print("Schema do DataFrame df_clima:")
df_clima.printSchema()
#print("\nAlgumas linhas do DataFrame:")
#df.show()

df_clima.createOrReplaceTempView("clima")

df_resultado_sql = spark.sql("""
SELECT DISTINCT Latitude, Longitude
FROM clima
""")

print("Resultado Preenchimento dados de cidades:")
df_resultado_sql.show()

df_resultado_sql.write.mode("overwrite").parquet('../data/silver/coord_cidades')


spark.stop()

Schema do DataFrame df_clima:
root
 |-- Data: string (nullable = true)
 |-- Tempo: string (nullable = true)
 |-- Temperatura: float (nullable = true)
 |-- Precipitacao: float (nullable = true)
 |-- Umidade: float (nullable = true)
 |-- Vento: float (nullable = true)
 |-- Regiao: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- Estacao: string (nullable = true)
 |-- Latitude: float (nullable = true)
 |-- Longitude: float (nullable = true)
 |-- Altitude: float (nullable = true)
 |-- Ano: integer (nullable = true)

Resultado Preenchimento dados de cidades:


+----------+----------+
|  Latitude| Longitude|
+----------+----------+
|-10.476944|-49.629475|
|-17.785833|-50.981388|
|-8.6663885|-35.568054|
|  -29.8425|  -57.0825|
|-28.275555|-49.934723|
|-17.923622| -51.71747|
|-6.0330553|-44.233334|
|-27.085312| -52.63571|
| -5.911111|-42.718613|
|-26.819445|-50.985554|
|-28.704721|-51.870556|
| -3.843611|-50.638054|
|-13.038642|  -57.0922|
|-14.980278|-49.539444|
|-11.366944|-58.733055|
|    -15.31|-45.616665|
|-6.7566667|  -38.2275|
|-15.723056|-42.435833|
|-22.314444|-45.373055|
| -20.44444| -52.87583|
+----------+----------+
only showing top 20 rows


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

# Função para obter a cidade a partir da latitude e longitude
def obter_cidade(lat, lon, contador, tentativas=3):
    geolocator = Nominatim(user_agent="geoapi")
    for tentativa in range(tentativas):
        try:
            location = geolocator.reverse((lat, lon), exactly_one=True)
            if location and 'address' in location.raw:
                address = location.raw['address']
                return address.get('city') or address.get('town') or address.get('village') or address.get('municipality') or f'Cidade não encontrada {contador}'
        except GeocoderTimedOut:
            if tentativa < tentativas - 1:
                time.sleep(2)  # Espera 2 segundos antes de tentar novamente
            else:
                return f'Cidade não encontrada {contador} (timeout)'
    return f'Cidade não encontrada {contador}'

caminho_arquivo = '../data/silver/coord_cidades'

df = pd.read_parquet(caminho_arquivo)

# Transformar latitude e longitude para o padrão dos EUA (com ponto)
df['Latitude'] = df['Latitude'].astype(str).str.replace(',', '.').astype(float)
df['Longitude'] = df['Longitude'].astype(str).str.replace(',', '.').astype(float)

# Obter latitudes e longitudes distintas
coords_distintas = df[['Latitude', 'Longitude']].drop_duplicates()

qtd_distintas = coords_distintas.count()
#print(qtd_distintas)

# Adicionar a coluna 'Cidade' ao DataFrame de coordenadas distintas
contador = 1
def obter_cidade_com_contador(row):
    global contador
    cidade = obter_cidade(row['Latitude'], row['Longitude'], contador)
    if 'Cidade não encontrada' in cidade:
        contador += 1
    return cidade

coords_distintas['Cidade'] = coords_distintas.apply(obter_cidade_com_contador, axis=1)

# Salvar o DataFrame de coordenadas distintas com cidades em um arquivo Parquet
coords_distintas.to_parquet(r'../data/silver/coords_cidades.parquet')

Latitude     1482
Longitude    1482
dtype: int64
